# 원본 라벨·비라벨을 제품별로 분리

data/original의 원본에서 PART_NAME이 CN7 또는 RG3로 시작하는 행을 각각 추출합니다. 대회용 가공 파일을 사용하지 않습니다. 원본 컬럼·센서값·Timestamp·RH/LH 행을 보존하며 정상/불량 필터링을 하지 않습니다. 출력 폴더: data/original_split.

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent
source = root / 'data/original'
output = root / 'data/original_split'
output.mkdir(parents=True, exist_ok=True)
summary = []
for kind, source_name in [('labeled', 'moldset_labeled.csv'), ('unlabeled', 'unlabeled_data.csv')]:
    pieces = {'CN7': [], 'RG3': []}
    for chunk in pd.read_csv(source / source_name, chunksize=100_000):
        for product in pieces:
            selected = chunk.loc[chunk['PART_NAME'].fillna('').str.startswith(product)].copy()
            if len(selected):
                pieces[product].append(selected)
    for product, frames in pieces.items():
        assert frames, (kind, product)
        result = pd.concat(frames, ignore_index=True)
        result['TimeStamp'] = pd.to_datetime(result['TimeStamp'], errors='raise')
        result = result.sort_values(['TimeStamp', 'Unnamed: 0']).reset_index(drop=True)
        assert result['PART_NAME'].str.startswith(product).all()
        assert result['TimeStamp'].notna().all()
        target = output / ('moldset_' + kind + '_' + product.lower() + '.csv')
        result.to_csv(target, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d %H:%M:%S')
        row = {'kind': kind, 'product': product, 'rows': len(result),
               'unique_timestamps': result['TimeStamp'].nunique(),
               'start': result['TimeStamp'].min(), 'end': result['TimeStamp'].max()}
        if kind == 'labeled':
            row['bad_rows'] = int(result['PassOrFail'].eq(1).sum())
            row['bad_shots'] = result.loc[result['PassOrFail'].eq(1), 'TimeStamp'].nunique()
        summary.append(row)
        print(row)
pd.DataFrame(summary).to_csv(root / 'reports/tables/original_product_split_counts.csv', index=False, encoding='utf-8-sig')
assert summary[0]['rows'] == 1425
assert summary[1]['rows'] == 1182
print('Original rows and sensor scale preserved. LH/RH rows are not combined.')